# Lab 5: Exponential weights and off-policy evaluation

The first part implements Hedge with full information and compares it with follow the
leader. The second part moves to bandit feedback with EXP3, which estimates the gains of
the arms that were not played by importance weighting, and compares it with UCB1 on a
stochastic bandit and on a sequence of rewards built against UCB1. The third part
evaluates recommendation policies from the logs of another policy: naive estimator,
importance sampling (IPS) and clipped IPS.

Replace the lines `...  # TODO` with your code. Each implementation is followed by a cell of
automatic tests, which must print "all tests passed".

This lab needs the version of `seqlearn` that contains `adversarial.py` and
`offpolicy.py`: if the imports below fail, download the setup archive again from the course
page and replace the folder `seqlearn/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from seqlearn import Agent, BernoulliBandit, UCBAgent, plot_mean, pseudo_regret, run, run_many
from seqlearn import checks
from seqlearn.adversarial import SequenceBandit, alternating_gains, regret_vs_best_arm, stochastic_gains
from seqlearn.offpolicy import (N_ITEMS, click_probabilities, collect_logs, constant_policy, draw_profiles,
                                old_recommender, softmax, true_value)
from seqlearn.plotting import COLORS, MUTED, set_style

set_style()

# Part I: Prediction with expert advice

## 1. Hedge

There are $K$ experts. At each round $t$ the agent chooses a probability vector $p_t$ over
the experts, then the gains $g_t \in [0, 1]^K$ of all the experts are revealed (full
information), and the agent receives $\langle p_t, g_t \rangle$. No assumption is made on
the gains: they can be any sequence. The regret compares with the best expert in
hindsight,
$$R_T = \max_a \sum_{t=1}^T g_t(a) - \sum_{t=1}^T \langle p_t, g_t \rangle \enspace.$$

Follow the leader (FTL) puts all the weight on the expert with the largest cumulative gain
$G_{t-1}(a) = \sum_{s < t} g_s(a)$. Hedge, with learning rate $\eta > 0$, plays
$$p_t(a) = \frac{\exp(\eta G_{t-1}(a))}{\sum_b \exp(\eta G_{t-1}(b))} \enspace,$$
the softmax of the cumulative gains with temperature $1/\eta$.

**Question 1.** Complete `hedge_probs(cum_gains, eta)`, which returns $p_t$ given the
array of the cumulative gains $G_{t-1}$.

*Hint:* after $1000$ rounds, $\exp(\eta G)$ overflows. Subtracting $\max_a \eta G(a)$ from
all the exponents does not change the probabilities.

In [ ]:
def hedge_probs(cum_gains, eta):
    ...  # TODO


checks.check_hedge(hedge_probs)

The function below plays a strategy on a table of gains of shape $(T, K)$ and returns the
gains $\langle p_t, g_t \rangle$ received at each round. A strategy is a function that maps
the cumulative gains to $p_t$.

In [ ]:
def play_full_information(strategy, gains):
    cum_gains = np.zeros(gains.shape[1])
    received = np.empty(len(gains))
    for t, g in enumerate(gains):
        p = strategy(cum_gains)
        received[t] = p @ g
        cum_gains += g
    return received


def follow_the_leader(cum_gains):
    return np.eye(len(cum_gains))[np.argmax(cum_gains)]


T_EXPERTS = 1000

**Question 2.** `alternating_gains(T)` returns the gains $(1/2, 0)$ at the first round,
then $(0, 1)$, $(1, 0)$, $(0, 1)$, and so on (Tutorial 5, exercise 1). Plot the regret of
FTL and of Hedge with $\eta = \sqrt{8 \log(K) / T}$ as functions of $t$, with
`regret_vs_best_arm(gains, received)`, for $T = 1000$.

In [ ]:
...  # TODO

**Question 3.** For $\eta \in \{0.003, 0.01, 0.03, 0.1, 0.3, 1, 3\}$, compute the final
regret of Hedge on the alternating gains, and its mean final regret over $20$ sequences of
stochastic gains `stochastic_gains([0.6, 0.4], T, rng=seed)`. Plot both as functions of
$\eta$, with a logarithmic x-axis, and add the final regret of FTL on both sequences as
horizontal lines.

In [ ]:
...  # TODO

# Part II: Bandit feedback

## 2. EXP3

With bandit feedback, only the reward $X_t$ of the arm $A_t$ that was played is observed.
EXP3 draws $A_t$ from $p_t$ and replaces the unknown loss $1 - x_t(a)$ of each arm by the
importance-weighted estimate
$$\hat\ell_t(a) = \frac{(1 - X_t)\, \mathbf{1}\{A_t = a\}}{p_t(a)} \enspace,$$
which is unbiased (Tutorial 5, exercise 2). It then plays Hedge on the estimated losses:
$p_t(a) \propto \exp(-\eta \hat L_{t-1}(a))$, where $\hat L_{t-1}(a) = \sum_{s < t}
\hat\ell_s(a)$. With $\eta = \sqrt{2 \log(K) / (TK)}$, the regret against the best arm is at
most $\sqrt{2 T K \log K}$, for any sequence of rewards in $[0, 1]$.

**Question 4.** Complete `EXP3Agent`. `probs()` returns $p_t$, `act()` stores it in
`self.p` and draws the arm, and `update(arm, reward)` updates `self.loss_hat`, the array
of the $\hat L(a)$.

In [ ]:
class EXP3Agent(Agent):
    def __init__(self, n_arms, eta, rng=None):
        self.eta = eta
        super().__init__(n_arms, rng)

    def reset(self):
        super().reset()
        self.loss_hat = np.zeros(self.n_arms)

    def probs(self):
        ...  # TODO

    def act(self):
        ...  # TODO

    def update(self, arm, reward):
        super().update(arm, reward)
        ...  # TODO


checks.check_exp3(EXP3Agent)

**Question 5.** On the stochastic bandit with Bernoulli arms of means $(0.6, 0.5, 0.4)$ and
$T = 5000$, plot the mean pseudo-regret of UCB1 and of EXP3 with
$\eta = \sqrt{2 \log(K)/(TK)}$ over $50$ runs, and print the mean final regrets
$\pm 2$ standard errors. Then print the mean final regrets for $T = 20000$, over $20$
runs, with the learning rate of EXP3 recomputed for this horizon.

In [ ]:
MEANS = [0.6, 0.5, 0.4]
HORIZON = 5000
N_RUNS = 50
eta_exp3 = np.sqrt(2 * np.log(len(MEANS)) / (HORIZON * len(MEANS)))


def final_stats(final):
    return final.mean(), 2 * final.std(ddof=1) / np.sqrt(len(final))

In [ ]:
...  # TODO

## 3. A sequence built against UCB1

UCB1 is deterministic: given the past rewards, its next arm is known. One can therefore
build in advance a sequence of rewards that it plays badly: simulate UCB1, and at each
round give reward $0$ to the arm it chooses and $1$ to the others.

**Question 6.** Build this table of rewards `trap`, of shape $(T, 3)$, with $T = 5000$, by
running a `UCBAgent(3)` and feeding it the reward $0$ at each round. Then play UCB1 and EXP3
($10$ runs, different seeds) on `SequenceBandit(trap)` with `run`, and plot their regret
against the best fixed arm with `regret_vs_best_arm(trap, rewards)`.

In [ ]:
...  # TODO

# Part III: Off-policy evaluation

## 4. The problem

We come back to the recommendation problem of the track `users` (Lab 4), with $5$ items and
$4$ moods. A recommender is in production: `old_recommender`. It has recorded logs: for
each user, the profile, the recommended item, the click (0 or 1), and the
**propensity**, the probability with which it chose this item. We want the click rate that
another policy would obtain, without deploying it.

A policy is a function that maps an array of profiles, of shape $(n, 4)$, to the array of
the probabilities of the items, of shape $(n, 5)$. `true_value(policy)` computes its click
rate by simulation: it is unknown in practice and only serves to evaluate the estimators.

In [ ]:
logs = collect_logs(old_recommender, 1000, rng=0)
print({key: value[:3].round(3) for key, value in logs.items()})
print(f"click rate of the old recommender: {true_value(old_recommender):.3f}")
print("frequency of each item in the logs:", np.bincount(logs["items"], minlength=N_ITEMS) / 1000)


def main_mood(profiles):
    """Recommend item j to the users whose main mood is j."""
    return np.eye(N_ITEMS)[np.argmax(profiles, axis=1)]


print(f"click rate of main_mood: {true_value(main_mood):.3f}")

**Question 7.** The naive estimator of the value of a deterministic policy $\pi$ is the
click rate on the logs where the recommended item is the one $\pi$ would have chosen.
Write `naive(logs, policy)`, then compute it for `main_mood` on $200$ sets of $1000$ logs
(`collect_logs(old_recommender, 1000, rng=seed)`), and compare the mean of the estimates
with the true value.

In [ ]:
...  # TODO

## 5. Importance sampling

The IPS estimator reweights each log by the ratio of the probabilities of the recommended
item under $\pi$ and under the logging policy $\mu$:
$$\hat V_{\mathrm{IPS}}(\pi) = \frac1n \sum_{i=1}^n \frac{\pi(a_i \mid u_i)}{\mu(a_i \mid u_i)}
\, r_i \enspace.$$
It is unbiased as soon as $\mu(a \mid u) > 0$ whenever $\pi(a \mid u) > 0$ (Tutorial 5,
exercise 2). The clipped estimator replaces the weight $w_i$ by $\min(w_i, M)$.

**Question 8.** Complete `ips(logs, policy, clip=np.inf)`.

In [ ]:
def ips(logs, policy, clip=np.inf):
    ...  # TODO


checks.check_ips(ips)

**Question 9.** On the $200$ datasets of Question 7, compute the IPS estimates of the value
of `main_mood` and of the policies `constant_policy(a)`, which always recommend item $a$,
for $a = 0, \ldots, 4$. For each policy, print the true value, the mean and the standard
deviation of the estimates, and the frequency of the item $a$ in the logs. Show the
histograms of the estimates for `main_mood`, `constant_policy(0)` and
`constant_policy(4)`.

In [ ]:
...  # TODO

**Question 10.** For `constant_policy(0)`, plot the bias, the standard deviation and the
root mean squared error (RMSE) of the clipped estimator over the $200$ datasets, as
functions of $M \in \{1, 2, 5, 10, 20, 50, 100, 1000\}$, with a logarithmic x-axis.

In [ ]:
...  # TODO

## Going further

**Question 11.** The self-normalized estimator divides by the sum of the weights instead
of $n$: $\hat V_{\mathrm{SN}} = \sum_i w_i r_i / \sum_i w_i$. Compare its bias and its
RMSE with those of IPS for `constant_policy(0)` and `main_mood`.

In [ ]:
...  # TODO

**Question 12.** Among the policies `softmax(profiles @ APPEAL_HAT.T, temperature=tau)`,
where `APPEAL_HAT` is the matrix of click rates estimated by least squares on the logs (one
regression per item, of the clicks on the profiles), choose the temperature
$\tau \in \{0.01, 0.03, 0.1, 0.3, 1\}$ with the best IPS estimate on one set of $5000$
logs, then compare with the true values.

In [ ]:
...  # TODO